In [3]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from ipywidgets import FloatSlider, Button, Label, HBox, VBox
from IPython.display import display
from vtkmodules.util import numpy_support
import vtk

DATA_FILE = "mixture.vti"

def load_vti(path):
    r = vtk.vtkXMLImageDataReader()
    r.SetFileName(path)
    r.Update()
    img = r.GetOutput()
    arr = img.GetPointData().GetArray(0)
    vol = numpy_support.vtk_to_numpy(arr)
    dims = img.GetDimensions()
    return vol.reshape(dims, order="F"), dims

vol, dims = load_vti(DATA_FILE)
vals = vol.flatten()
vmin, vmax = float(vals.min()), float(vals.max())

x = np.arange(dims[0]); y = np.arange(dims[1]); z = np.arange(dims[2])
x, y, z = np.meshgrid(x, y, z, indexing="ij")

def plasma_color(v):
    t = np.clip((v-vmin)/(vmax-vmin),0,1)
    c = px.colors.sample_colorscale("Plasma",[t])[0]
    return [[0,c],[1,c]]

iso = go.FigureWidget([go.Isosurface(
    x=x.flatten(), y=y.flatten(), z=z.flatten(),
    value=vals,
    isomin=0.0,
    isomax=0.0,
    surface_count=1,
    colorscale=plasma_color(0.0),
    showscale=False,
    caps=dict(x_show=False,y_show=False,z_show=False)
)])
iso.update_layout(
    title="Isosurface",
    width=550,height=550,
    margin=dict(l=0,r=0,b=0,t=40),
    scene=dict(xaxis_title="x",yaxis_title="y",zaxis_title="z")
)

hist = go.FigureWidget([go.Histogram(x=vals, nbinsx=50)])
hist.update_layout(
    title="Histogram",
    width=550,height=550,
    margin=dict(l=0,r=0,b=0,t=40),
    xaxis_title="Vortex scalar values",
    yaxis_title="Frequency"
)

slider = FloatSlider(
    value=0.0,
    min=vmin,
    max=vmax,
    step=0.01,
    description="Isoval:",
    continuous_update=True,
    readout=False
)
value_label = Label("0.00")
reset = Button(description="Reset")

def update(v):
    value_label.value = f"{v:.2f}"
    with iso.batch_update():
        iso.data[0].isomin = v
        iso.data[0].isomax = v
        iso.data[0].colorscale = plasma_color(v)

    subset = vals[(vals >= v-0.25) & (vals <= v+0.25)]

    with hist.batch_update():
        hist.data[0].x = subset
        hist.data[0].nbinsx = 50
        hist.update_xaxes(range=[v-0.25, v+0.25])
        hist.update_yaxes(autorange=True)

slider.observe(lambda c: update(c["new"]), names="value")

def reset_cb(_):
    slider.value = 0.0
    value_label.value = "0.00"
    with iso.batch_update():
        iso.data[0].isomin = 0.0
        iso.data[0].isomax = 0.0
        iso.data[0].colorscale = plasma_color(0.0)
    with hist.batch_update():
        hist.data[0].x = vals
        hist.data[0].nbinsx = 50
        hist.update_xaxes(range=[vmin, vmax])
        hist.update_yaxes(autorange=True)

reset.on_click(reset_cb)

display(VBox([
    HBox([slider, value_label, reset]),
    HBox([iso, hist])
]))
